# Aula 06: Lógica de Predicados, Quantificadores e Módulo de Varredura Global de Sensores

**Projeto:** SCADA-Core Automática / Planta de Fabricação de Paçoca (Grupo 7)

---

Nesta aula, implementamos a **Lógica de Primeira Ordem (Lógica de Predicados)** para construir o **Módulo de Varredura Global de Sensores** (*SCADA State Scanning Engine*), permitindo avaliar sentenças universais ($\forall$) e existenciais ($\exists$) sobre o ecossistema de instrumentos da nossa planta de paçoca.

In [ ]:
import time
import random
from enum import Enum
from dataclasses import dataclass, field
from typing import List, Dict, Callable, Tuple, Any, Optional

try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

def exibir_tabela(dados, titulo=""):
    if titulo:
        print(f"\n=== {titulo} ===")
    if HAS_PANDAS:
        display(pd.DataFrame(dados)) if 'display' in globals() else print(pd.DataFrame(dados))
    else:
        if isinstance(dados, dict):
            for k, v in dados.items():
                print(f"{k:30}: {v}")
        elif isinstance(dados, list):
            for item in dados:
                print(item)

print("Ambiente de desenvolvimento da Aula 06 inicializado com sucesso.")

## 1. Modelagem das Entidades da Planta de Paçoca

In [ ]:
class Setor(Enum):
    SETOR_100 = "Setor 100 - Recepção e Limpeza"
    SETOR_200 = "Setor 200 - Torra e Despeliculagem"
    SETOR_400 = "Setor 400 - Compactação e Embalagem"

class TipoInstrumento(Enum):
    TRANSMISSOR_TEMPERATURA = "TT"
    DETECTOR_METAL = "MD"
    CAMERA_VISAO = "VS"
    PRESSOSTATO = "PS"
    VALVULA_BLOQUEIO = "XV"
    MOTOR = "M"
    EMERGENCIA = "ESD"

@dataclass
class Instrumento:
    tag: str
    tipo: TipoInstrumento
    setor: Setor
    descricao: str
    online: bool = True
    calibrado: bool = True
    valor_atual: float = 0.0
    limite_critico_alto: Optional[float] = None
    fim_de_curso_aberto: Optional[bool] = None
    motor_ligado: Optional[bool] = None
    emergencia_ativa: Optional[bool] = None
    metal_detectado: Optional[bool] = None
    defeito_optico: Optional[bool] = None

def criar_parque_instrumentos() -> List[Instrumento]:
    return [
        # Setor 100
        Instrumento("ESD-100", TipoInstrumento.EMERGENCIA, Setor.SETOR_100, "Botão Emergência Recepção", emergencia_ativa=False),
        Instrumento("M-101", TipoInstrumento.MOTOR, Setor.SETOR_100, "Peneira Vibratória", motor_ligado=True),

        # Setor 200
        Instrumento("TT-201", TipoInstrumento.TRANSMISSOR_TEMPERATURA, Setor.SETOR_200, "Temperatura Forno de Torra", valor_atual=145.0, limite_critico_alto=160.0),
        Instrumento("XV-201", TipoInstrumento.VALVULA_BLOQUEIO, Setor.SETOR_200, "Válvula Gás Queimador", fim_de_curso_aberto=True),
        Instrumento("M-201", TipoInstrumento.MOTOR, Setor.SETOR_200, "Esteira do Forno", motor_ligado=True),

        # Setor 400
        Instrumento("PS-402", TipoInstrumento.PRESSOSTATO, Setor.SETOR_400, "Pressão Rede Pneumática", valor_atual=6.5),
        Instrumento("MD-401", TipoInstrumento.DETECTOR_METAL, Setor.SETOR_400, "Detector de Metais Embalagem", metal_detectado=False),
        Instrumento("VS-401", TipoInstrumento.CAMERA_VISAO, Setor.SETOR_400, "Câmera IA Inspeção", defeito_optico=False),
        Instrumento("XV-401", TipoInstrumento.VALVULA_BLOQUEIO, Setor.SETOR_400, "Válvula Rejeição Sopro", fim_de_curso_aberto=False),
        Instrumento("M-401", TipoInstrumento.MOTOR, Setor.SETOR_400, "Esteira de Embalagem", motor_ligado=True),
    ]

instrumentos_base = criar_parque_instrumentos()
print(f"Total de instrumentos cadastrados na planta de paçoca: {len(instrumentos_base)}")

## 2. Motor Algorítmico de Quantificação ($\forall$ e $\exists$)

In [ ]:
def forall(dominio: List[Any], predicado: Callable[[Any], bool]) -> Tuple[bool, List[Any]]:
    contraexemplos = [x for x in dominio if not predicado(x)]
    return (len(contraexemplos) == 0, contraexemplos)

def exists(dominio: List[Any], predicado: Callable[[Any], bool]) -> Tuple[bool, List[Any]]:
    testemunhas = [x for x in dominio if predicado(x)]
    return (len(testemunhas) > 0, testemunhas)

def forall_in(dominio: List[Any], guarda: Callable[[Any], bool], predicado: Callable[[Any], bool]) -> Tuple[bool, List[Any]]:
    return forall([x for x in dominio if guarda(x)], predicado)

def exists_in(dominio: List[Any], guarda: Callable[[Any], bool], predicado: Callable[[Any], bool]) -> Tuple[bool, List[Any]]:
    return exists([x for x in dominio if guarda(x)], predicado)

## 3. Definição dos Predicados Operacionais da Fábrica

In [ ]:
def is_saudavel(inst: Instrumento) -> bool:
    return inst.online and inst.calibrado

def is_emergencia_ativa(inst: Instrumento) -> bool:
    return inst.emergencia_ativa is True

def is_metal_detectado(inst: Instrumento) -> bool:
    return inst.metal_detectado is True

def is_defeito_optico(inst: Instrumento) -> bool:
    return inst.defeito_optico is True

def is_sobretemperatura(inst: Instrumento) -> bool:
    if inst.tipo == TipoInstrumento.TRANSMISSOR_TEMPERATURA and inst.limite_critico_alto is not None:
        return inst.valor_atual > inst.limite_critico_alto
    return False

def is_pressao_ar_ok(inst: Instrumento) -> bool:
    if inst.tipo == TipoInstrumento.PRESSOSTATO:
        return inst.valor_atual > 6.0
    return True

def is_valvula_aberta(inst: Instrumento) -> bool:
    return inst.fim_de_curso_aberto is True

def is_motor_ligado(inst: Instrumento) -> bool:
    return inst.motor_ligado is True


## 4. Implementação do Módulo de Varredura Global (`SCADAScanningEngine`)

In [ ]:
class SCADAScanningEngine:
    def __init__(self, instrumentos: List[Instrumento]):
        self.instrumentos = instrumentos

    def executar_varredura(self) -> Dict[str, Any]:
        # 1. Prontidão Geral
        univ_saudavel, falhas = forall(self.instrumentos, is_saudavel)
        emergencia, _ = exists_in(self.instrumentos, lambda i: i.tipo == TipoInstrumento.EMERGENCIA, is_emergencia_ativa)
        plant_readiness = univ_saudavel and not emergencia

        # 2. Intertrava de Qualidade
        metal, _ = exists_in(self.instrumentos, lambda i: i.tipo == TipoInstrumento.DETECTOR_METAL, is_metal_detectado)
        defeito, _ = exists_in(self.instrumentos, lambda i: i.tipo == TipoInstrumento.CAMERA_VISAO, is_defeito_optico)
        quality_trip = metal or defeito

        # 3. Alívio Térmico Forno
        thermal_trip, test_temp = exists_in(self.instrumentos, lambda i: i.setor == Setor.SETOR_200, is_sobretemperatura)

        # 4. Permissivo Embalagem
        pressoes_ok, _ = forall_in(self.instrumentos, lambda i: i.tipo == TipoInstrumento.PRESSOSTATO, is_pressao_ar_ok)
        motores_ok, _ = exists_in(self.instrumentos, lambda i: i.setor == Setor.SETOR_400 and i.tipo == TipoInstrumento.MOTOR, is_motor_ligado)
        perm_embalagem = pressoes_ok and motores_ok

        return {
            "PlantReadiness_OK": plant_readiness,
            "QualityTrip_Ativo": quality_trip,
            "ThermalTrip_Forno_Ativo": thermal_trip,
            "Permissivo_Embalagem_OK": perm_embalagem
        }

engine = SCADAScanningEngine(instrumentos_base)
exibir_tabela(engine.executar_varredura(), "Estado Nominal da Fábrica de Paçoca")

## 5. Simulação de Cenários Críticos

In [ ]:
cenarios = {}

# Cenário A: Metal Detectado
inst_cen_a = criar_parque_instrumentos()
inst_cen_a[5].metal_detectado = True  # MD-401
cenarios["1. Metal Detectado na Esteira"] = SCADAScanningEngine(inst_cen_a).executar_varredura()

# Cenário B: Sobreaquecimento Forno
inst_cen_b = criar_parque_instrumentos()
inst_cen_b[2].valor_atual = 175.0  # TT-201 > 160.0
cenarios["2. Forno em Sobreaquecimento"] = SCADAScanningEngine(inst_cen_b).executar_varredura()

if HAS_PANDAS:
    df_cenarios = pd.DataFrame(cenarios)
    display(df_cenarios) if 'display' in globals() else print(df_cenarios)
else:
    for nome, res in cenarios.items():
        exibir_tabela(res, nome)